In [ ]:
'''
task: classify syllogism validity with CLIF notation
models: gemma-2-2b-it, llama-3.2-3b-instruct, phi-3.5-mini-instruct
dataset: pfolio
evaluation: zero-shot + sef category
'''
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd

pfolio_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_kr_gold_train_sef.csv")

In [ ]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, sef, ref_relation=None, source_knowledge=None):
  # define sef categories
  sef_disjunctive = r"""
  A disjunctive syllogism contains "∨" or "⊕". Here is an example:
  <PREMISES>forall x (kid(x) implies young(x))
forall x (toddler(x) implies kid(x))
forall x (young(x) implies not elderly(x))
forall x (pirate(x) implies seafarer(x))
not pirate(nancy) implies young(nancy)
not toddler(nancy) implies seafarer(nancy)</PREMISES>
  <CONCLUSION>not (pirate(nancy) xor toddler(nancy))</CONCLUSION>
  """

  sef_complex = r"""
  A complex syllogism has more than 2 premises. Here is an example:
  <PREMISES>woncup(aberdeen, year2013final)
woncup(rangers, year2014final)
not (aberdeen=rangers)
forall x forall y forall z forall w (not (x=y) and woncup(x, z) and woncup(y, w) implies not (z=w))</PREMISES>
  <CONCLUSION>exists x (woncup(aberdeen, x))</CONCLUSION>
  """

  sef_categorical = r"""
  A categorical syllogism contains any word from the list ["all", "any", "some", "no", "few", "most", "none", "several"]. Here is an example:
  <PREMISES>unincorporatedcommunity(ordinary)
locatedin(ordinary, elliotcounty) and on(ordinary, kentuckyroute32)
locatednorthwestof(ordinary, sandyhook)</PREMISES>
  <CONCLUSION>exists x (unincorporatedcommunity(x) and locatedin(x, elliotcounty))</CONCLUSION>
  """

  sef_hypothetical = r"""
  A hypothetical syllogism is not disjunctive, complex or categorical. Here is an example:
  <PREMISES>forall x (homework(x) implies not fun(x))
exists x (reading(x) and homework(x))</PREMISES>
  <CONCLUSION>exists x (reading(x) and fun(x))</CONCLUSION>
  """

  sef_categories = dict()
  sef_categories["disjunctive"] = sef_disjunctive
  sef_categories["complex"] = sef_complex
  sef_categories["categorical"] = sef_categorical
  sef_categories["hypothetical"] = sef_hypothetical


  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline | keyword* quantifier* symbol* leftparen* (quantifier symbol)* proposition rightparen* newline | keyword* (quantifier symbol)* leftparen* (quantifier symbol)* proposition rightparen* newline
    proposition: atomicproposition | complexproposition
    complexproposition: keyword* proposition keyword leftparen* (quantifier symbol)* proposition rightparen*
    atomicproposition: leftparen* term* leftparen* term* rightparen*
    !term: (LETTER+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")* | (DIGIT+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠")*
    !leftparen: "("
    !rightparen: ")"
    !keyword: "and" | "not" | "implies" | "or" | "xor" | "implies" | "implies"
    !quantifier: "exists" | "forall"
    symbol: LETTER
    newline: /\n/

    %import common.LETTER
    %import common.DIGIT
    %import common.INT -> NUMBER
    %import common.ESCAPED_STRING -> STRING
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in CLIF with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  The CLIF BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  You are also given the category of the syllogism to help you understand it: {sef_categories[sef]}.
  Classify the conclusion as "T" if true, "F" if false or "U" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

In [ ]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      conclusion = row["Conclusions - " + notation]
      premises = row["Premises - " + notation]
      sef_category = row["sef"]
      label = row["Truth Values"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, sef_category, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

In [ ]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CLIF')

*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 ocellatedwildturkey(tom)
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 easternwildturkey(tom)
*** True Label: 
 F
*** Predicted Label: 
 T
*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(

In [ ]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.47840531561461797
***** PRECISION *****
0.4422816977561503
***** RECALL *****
0.44381088144452363
***** F1 *****
0.3686868686868687


,Accuracy,Precision,Recall,F1
0,0.478405,0.442282,0.443811,0.368687


In [ ]:
# try rag search with llama
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CLIF')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 ocellatedwildturkey(tom)
*** True Label: 
 T
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 easternwildturkey(tom)
*** True Label: 
 F
*** Predicted Label: 
 T


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

*** Premises: 
 american(maggiefriedman) and screenwriter(maggiefriedman) and producer(maggiefriedman)
showrunnerof(maggiefriedman, witchesofeastend) and executiveproducerof(maggiefriedman, witchesofeastend) and lifetimetelevisionseries(maggiefriedman)
fantasydrama(witchesofeastend) and series(witchesofeastend)
produces(maggiefriedman, eastwick) and develops(maggiefriedman, eastwick)
series(eastwick) and airedon(eastwick, abc)
*** Conclusion: 
 develops(maggiefriedman, witchesofeastend)
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 largecomplex(shafaq-asiman) and largecomplex(shafaq-asiman) and offshore(shafaq-asiman) and geologicalstructures(shafaq-asiman) and in(shafaq-asiman, caspiansea)
northwestof(baku, shafaq-asiman)
forall x forall y (northwestof(x, y) implies southeastof(y, x))
*** Conclusion: 
 southeastof(baku, shafaq-asiman)
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 largecomplex(shafaq-asiman) and largecomplex(shafaq-asiman) and offshore(shafaq-asiman) and geologicalstructures(shafaq-asiman) and in(shafaq-asiman, caspiansea)
northwestof(baku, shafaq-asiman)
forall x forall y (northwestof(x, y) implies southeastof(y, x))
*** Conclusion: 
 exists x (largecomplex(x) and southeastof(x, baku))
*** True Label: 
 T
*** Predicted Label: 
 None
*** Premises: 
 largecomplex(shafaq-asiman) and largecomplex(shafaq-asiman) and offshore(shafaq-asiman) and geologicalstructures(shafaq-asiman) and in(shafaq-asiman, caspiansea)
northwestof(baku, shafaq-asiman)
forall x forall y (northwestof(x, y) implies southeastof(y, x))
*** Conclusion: 
 forall x (geologicalstructures(x) and offshore(x) implies not northwestof(baku, x))
*** True Label: 
 F
*** Predicted Label: 
 None
Classification Report:               precision    recall  f1-score   support

                   0.00      0.00      0.00         0
           F       0.43      0.04      0.08       

In [ ]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.026578073089700997
***** PRECISION *****
0.08767951625094482
***** RECALL *****
0.012065981060991752
***** F1 *****
0.02086592319725488


,Accuracy,Precision,Recall,F1
0,0.026578,0.08768,0.012066,0.020866


In [ ]:
# try rag search with phi
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", device_map="auto")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='CLIF')

*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 ocellatedwildturkey(tom)
*** True Label: 
 T
*** Predicted Label: 
 F</output>
*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or gouldswildturkey(x) or merriamswildturkey(x) or riograndewildturkey(x) or ocellatedwildturkey(x)))
not (easternwildturkey(tom))
not (osceolawildturkey(tom))
not (gouldswildturkey(tom))
not (merriamswildturkey(tom) or riograndewildturkey(tom))
wildturkey(tom)
*** Conclusion: 
 easternwildturkey(tom)
*** True Label: 
 F
*** Predicted Label: 
 F</output>
*** Premises: 
 forall x (wildturkey(x) implies (easternwildturkey(x) or osceolawildturkey(x) or

In [ ]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.23588039867109634
***** PRECISION *****
0.25078285282366913
***** RECALL *****
0.1072846374343167
***** F1 *****
0.14156317996630002


,Accuracy,Precision,Recall,F1
0,0.23588,0.250783,0.107285,0.141563


In [ ]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)